# Relation Extraction & Knowledge Graph Construction Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: pattern-based extraction

In [ ]:
```python

PATTERNS = [

    (r"(?P<s>[A-Z]\w+) (?:is|was) (?:a|an|the) (?P<o>[A-Z]?\w+)", "isA"),

    (r"(?P<s>[A-Z]\w+) (?:is|was) born in (?P<o>\w+)", "bornIn"),

    (r"(?P<s>[A-Z]\w+) works? (?:at|for) (?P<o>[A-Z]\w+)", "worksAt"),

    (r"(?P<s>[A-Z]\w+) founded (?P<o>[A-Z]\w+)", "founded"),

]

In [ ]:
```

See `code/main.py` for the full toy extractor. Hearst patterns still ship in domain-specific pipelines because they are debuggable.

### Step 2: supervised relation classification

In [ ]:
```python

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained("Babelscape/rebel-large")

model = AutoModelForSequenceClassification.from_pretrained("Babelscape/rebel-large")

text = "Tim Cook was born in Alabama. He later became CEO of Apple."

encoded = tok(text, return_tensors="pt", truncation=True)

output = model.generate(**encoded, max_length=200)

triples = tok.batch_decode(output, skip_special_tokens=False)

In [ ]:
```

REBEL is a seq2seq relation extractor: text in, triples out, already in Wikidata property ids. Fine-tuned on distant-supervision data. Standard open-weights baseline.

### Step 3: LLM-prompted extraction with anchoring

In [ ]:
```python

prompt = f"""Extract (subject, relation, object) triples from the text.

For each triple, include the exact character span in the source text.

Text: {text}

Output JSON:

[{{"subject": {{"text": "...", "span": [start, end]}},

   "relation": "...",

   "object": {{"text": "...", "span": [start, end]}}}}, ...]

Only include triples fully supported by the text. No inference beyond what is stated.

"""

In [ ]:
```

Verify every returned span against the source. Reject anything where `text[start:end] != triple_entity`. This is the AEVS "verify" step in its minimal form.

### Step 4: canonicalize onto a closed ontology

In [ ]:
```python

RELATION_MAP = {

    "is the CEO of": "P169",       # "chief executive officer"

    "was born in":   "P19",         # "place of birth"

    "founded":        "P112",       # "founded by" (inverted subject/object)

    "works at":       "P108",       # "employer"

}

def canonicalize(relation):

    rel_low = relation.lower().strip()

    if rel_low in RELATION_MAP:

        return RELATION_MAP[rel_low]

    return None   # drop unmapped open relations or route to manual review

In [ ]:
```

Canonicalization is often 60-80% of the engineering work. Budget for it.

### Step 5: build a small graph and query

In [ ]:
```python

triples = extract(text)

graph = {}

for s, r, o in triples:

    graph.setdefault(s, []).append((r, o))

def neighbors(node, relation=None):

    return [(r, o) for r, o in graph.get(node, []) if relation is None or r == relation]

print(neighbors("Tim Cook", relation="P108"))    # -> [(P108, Apple)]

In [ ]:
```

This is the atom of every RAG-over-KG system. Scale it with RDF triple stores (Blazegraph, Virtuoso), property graphs (Neo4j), or vector-augmented graph stores.

## Exercises

In [ ]:
1. **Easy.** Run the pattern extractor in `code/main.py` on 5 news-article sentences. Hand-check precision.
2. **Medium.** Use REBEL (or a small LLM) on the same sentences. Compare triples. Which extractor has higher precision? Higher recall?
3. **Hard.** Build the AEVS pipeline: extract with LLM + verify spans against source. Measure hallucination rate before vs after the verify step on 50 Wikipedia-style sentences.